# Hexaly Planted Benchmarking

### Import libraries

In [ ]:
import time
import numpy as np
import hexaly.optimizer
import os

### Set up Hexaly solver

In [2]:
def hexaly_solver(Q,c,sum_constraint, time_limit= 1800):
    n = len(np.squeeze(c))
    def external_xTQx(x):
        return x @ Q @ x

    def external_2Qx(x):
        return 2 * Q @ x
    
    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        model = optimizer.model
        x = [model.float(0.0,100.0) for _ in range(n)]
        linear_term = model.sum(c[i]*x[i] for i in range(n))
        xTQx_func = model.create_double_external_function(external_xTQx, external_2Qx)
        objective = linear_term + xTQx_func(x)

        model.constraint(model.sum(x) == sum_constraint)
        model.minimize(objective)
        model.close()

        optimizer.param.time_limit = time_limit
        print("Starting Hexaly solver...")
        optimizer.solve()
        solution = [x[i].value for i in range(n)]
        objective_val = objective.value
        print(f"Status: {optimizer.solution.status}")
        return solution, objective_val


### load data file

In [3]:

def loadQ(path, n):
    tri = np.load(path, mmap_mode="r")
    Q = np.empty((n,n), dtype=tri.dtype)
    iu = np.triu_indices(n)
    Q[iu] = tri
    Q.T[iu] = tri
    return Q

### Solve with hexaly

In [4]:
# Instance details
optimal_energy = 12000
sum_constraint = 100
num_var = 2000
k = 44
ub =10
seed =100

name = f"STQP_n_{num_var}_k_{k}_R_{sum_constraint}_seed_{seed}_ub_{ub}"
instance_path = os.path.join(f"Instances/{name}.npy")
c = np.zeros(num_var)
try:
    Q= loadQ(instance_path,num_var)
    print("loaded file sucessfully.")
except FileNotFoundError:
    print(f"File {instance_path} does not exist.")


loaded file sucessfully.


In [5]:
# solver parameters
time_limit = 600

In [6]:
# solve with Hexaly
hexaly_start = time.time()
hex_sol_list, hex_obj = hexaly_solver(Q,c,sum_constraint,time_limit)
hexaly_end = time.time()
hex_sol = np.array(hex_sol_list)
hex_indices = np.where(hex_sol>1e-6)[0]

print(f"Number of non-zero nodes found by Hexaly:{len(hex_indices)}")
print(f"Eenergy value obtained by Hexaly:{hex_obj}")
print(f"Time taken by Hexaly:{hexaly_end- hexaly_start}")

Preprocess model 100%Starting Hexaly solver...
Push initial solution 100%
Model:  expressions = 4009, decisions = 2000, constraints = 1, objectives = 1
Param:  time limit = 600 sec, no iteration limit

[objective direction ]:     minimize

[  0 sec,       0 itr]: No feasible solution found (infeas = 1)
[  1 sec,       0 itr]: No feasible solution found (infeas = 1)
[  2 sec,     218 itr]:      22859.4
[ optimality gap     ]:      100.00%
[  3 sec,    1294 itr]:      15956.2
[  4 sec,    1519 itr]:      15811.3
[  5 sec,    1796 itr]:      15811.1
[  6 sec,    2012 itr]:        15811
[  7 sec,    2012 itr]:        15811
[  8 sec,    2464 itr]:      15810.6
[  9 sec,    2623 itr]:      15810.3
[ 10 sec,    2623 itr]:      15810.3
[ 11 sec,    2893 itr]:      15810.2
[ 12 sec,    3103 itr]:        15810
[ optimality gap     ]:      100.00%
[ 13 sec,    3431 itr]:      15809.6
[ 14 sec,    3431 itr]:      15809.6
[ 15 sec,    3588 itr]:      15809.3
[ 16 sec,    4014 itr]:      15809.1
[ 1

In [7]:
# compute relative gap
tol = 1e-7
abs_gap = hex_obj-optimal_energy
relative_gap = (round(abs_gap,6)*100)/round(optimal_energy,6)


In [8]:
# print result
if abs_gap>tol:
    print(f"Hexaly solution is not optimal")
    print(f"Absolute Gap:{abs_gap}")
    print(f"Relative Gap(%):{abs(relative_gap)}")
else:
    print(f"Optimal solution found:{hex_obj}.")

Hexaly solution is not optimal
Absolute Gap:0.020820610488954117
Relative Gap(%):0.00017350833333333333
